# Colab Drive Training Runner

This notebook mounts Google Drive, uploads and unzips `project.zip`, prepares the project to read data from Drive and write runs to Drive, and runs any shell script (for example `train.sh` or `train_smoke_run.sh`).

Drive layout used by default:
- Root: `/content/drive/MyDrive/PSBD`
- Data: `/content/drive/MyDrive/PSBD/preprocessed_data`
- Runs: `/content/drive/MyDrive/PSBD/runs`

In [ ]:
import os
import shutil
import subprocess
import time
from pathlib import Path

from google.colab import drive, files

In [ ]:
# ---------- Config ----------
ROOT_DRIVE_DIR = Path("/content/drive/MyDrive/PSBD")
DRIVE_DATA_DIR = ROOT_DRIVE_DIR / "preprocessed_data"
DRIVE_RUNS_DIR = ROOT_DRIVE_DIR / "runs"

# Where uploaded zip will be extracted
COLAB_WORK_DIR = Path("/content")
PROJECT_ZIP_NAME = "project.zip"

# Optional: set after unzip, e.g. PROJECT_DIR = Path('/content/PSBD-reproduce')
PROJECT_DIR = None

In [ ]:
def mount_google_drive(force_remount: bool = True) -> None:
    drive.mount("/content/drive", force_remount=force_remount)


def ensure_drive_layout(root_drive_dir: Path = ROOT_DRIVE_DIR) -> None:
    root_drive_dir.mkdir(parents=True, exist_ok=True)
    (root_drive_dir / "preprocessed_data").mkdir(parents=True, exist_ok=True)
    (root_drive_dir / "runs").mkdir(parents=True, exist_ok=True)
    print(f"Drive layout ready at: {root_drive_dir}")


def upload_and_unzip_project(
    zip_name: str = PROJECT_ZIP_NAME,
    work_dir: Path = COLAB_WORK_DIR,
    clear_existing_extracted_dir: bool = False,
) -> Path:
    work_dir.mkdir(parents=True, exist_ok=True)

    zip_path = work_dir / zip_name
    if not zip_path.exists():
        print(f"Upload {zip_name}...")
        uploaded = files.upload()
        if zip_name not in uploaded:
            raise FileNotFoundError(
                f"Expected upload named {zip_name}. Uploaded files: {list(uploaded.keys())}"
            )
        with open(zip_path, "wb") as f:
            f.write(uploaded[zip_name])

    extract_root = work_dir

    # Unzip project.zip in /content.
    # This assumes the zip contains one top-level project folder.
    subprocess.run(
        ["unzip", "-q", "-o", str(zip_path), "-d", str(extract_root)], check=True
    )

    candidates = [p for p in extract_root.iterdir() if p.is_dir() and p.name != "drive"]
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise RuntimeError("Could not find extracted project directory.")

    project_dir = candidates[0]

    if clear_existing_extracted_dir and project_dir.exists():
        # Not deleting by default to avoid accidental data loss.
        pass

    print(f"Using project directory: {project_dir}")
    return project_dir


def _replace_with_symlink(target_path: Path, link_path: Path) -> None:
    if link_path.is_symlink():
        if link_path.resolve() == target_path.resolve():
            return
        link_path.unlink()
    elif link_path.exists():
        backup_path = link_path.with_name(link_path.name + "_local_backup")
        if backup_path.exists():
            raise FileExistsError(f"Backup already exists: {backup_path}")
        shutil.move(str(link_path), str(backup_path))
        print(f"Moved existing {link_path.name} to backup: {backup_path}")

    link_path.symlink_to(target_path, target_is_directory=True)


def prepare_project_drive_links(
    project_dir: Path,
    drive_data_dir: Path = DRIVE_DATA_DIR,
    drive_runs_dir: Path = DRIVE_RUNS_DIR,
) -> None:
    ensure_drive_layout(ROOT_DRIVE_DIR)

    local_data_dir = project_dir / "preprocessed_data"
    local_runs_dir = project_dir / "runs"

    if local_data_dir.exists() and not local_data_dir.is_symlink():
        if not any(drive_data_dir.iterdir()):
            print(
                "Drive preprocessed_data is empty; copying local preprocessed_data once..."
            )
            for item in local_data_dir.iterdir():
                dst = drive_data_dir / item.name
                if item.is_dir():
                    shutil.copytree(item, dst, dirs_exist_ok=True)
                else:
                    shutil.copy2(item, dst)

    _replace_with_symlink(drive_data_dir, local_data_dir)
    _replace_with_symlink(drive_runs_dir, local_runs_dir)

    print(f"Linked {local_data_dir} -> {drive_data_dir}")
    print(f"Linked {local_runs_dir} -> {drive_runs_dir}")


def sync_runs_to_drive(
    project_dir: Path, drive_runs_dir: Path = DRIVE_RUNS_DIR
) -> None:
    local_runs_dir = project_dir / "runs"

    if local_runs_dir.is_symlink():
        # Already writing directly to Drive.
        return

    drive_runs_dir.mkdir(parents=True, exist_ok=True)
    if not local_runs_dir.exists():
        return

    for item in local_runs_dir.iterdir():
        dst = drive_runs_dir / item.name
        if item.is_dir():
            shutil.copytree(item, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dst)


def run_shell_script(
    project_dir: Path,
    script_name: str = "train.sh",
    sync_after_run: bool = True,
    env_overrides: dict | None = None,
) -> None:
    script_path = project_dir / script_name
    if not script_path.exists():
        raise FileNotFoundError(f"Script not found: {script_path}")

    env = os.environ.copy()
    if env_overrides:
        env.update({k: str(v) for k, v in env_overrides.items()})

    print(f"Running script: {script_path}")
    subprocess.run(
        ["bash", str(script_path)], cwd=str(project_dir), env=env, check=True
    )

    if sync_after_run:
        sync_runs_to_drive(project_dir, DRIVE_RUNS_DIR)
        print("Sync complete.")

In [ ]:
# ---------- One-time setup ----------
mount_google_drive(force_remount=True)
ensure_drive_layout(ROOT_DRIVE_DIR)
PROJECT_DIR = upload_and_unzip_project(
    zip_name=PROJECT_ZIP_NAME, work_dir=COLAB_WORK_DIR
)
prepare_project_drive_links(PROJECT_DIR, DRIVE_DATA_DIR, DRIVE_RUNS_DIR)
print("Setup done.")

In [ ]:
# ---------- Run any script ----------
# Example full run:
run_shell_script(PROJECT_DIR, script_name="train.sh")

# Example smoke run:
# run_shell_script(PROJECT_DIR, script_name='train_smoke_run.sh')

In [ ]:
# Optional quick check: list latest runs in Drive
runs = sorted(DRIVE_RUNS_DIR.glob("*"), key=lambda p: p.stat().st_mtime, reverse=True)
for p in runs[:10]:
    print(p)